# Notebook 1: Data Preparation
### Multimodal Driving Risk Prediction | MSc Business Analytics | Dublin Business School

---

## What This Notebook Does
Downloads and prepares the NVIDIA PhysicalAI Autonomous Vehicle Dataset for the research pipeline.
Raw clip metadata is filtered, clustered, and sampled to produce a clean, diverse analytical dataset.

## Key Steps
1. **Authenticate** with Hugging Face and download three metadata files
2. **Filter** clips: retain only valid clips with ego-motion and camera data present
3. **Cluster** clips using K-Means (k=4) to identify scenario groups
4. **Sample** 2,000 clips using diversity-aware strategy (1,000 rare + 1,000 normal)
5. **Engineer** contextual features: `is_night`, `is_peak_hour` from `hour_of_day`
6. **Save** prepared dataset to Google Drive

## Critical Design Decision
The Isolation Forest anomaly scores used during sampling are **deliberately excluded** from the output CSV.
They serve only to identify rare clips for download diversity: saving them would cause **target leakage** into Notebook 5.

## Input
- NVIDIA PhysicalAI Dataset via Hugging Face (authenticated access)

## Output
- `prepared_dataset_2000.csv`: 2,000 clips, clip-level metadata with contextual features
- `prepared_dataset_2000.parquet`: same dataset in parquet format

## Next Notebook
→ **Notebook 2** uses this output to extract ego-motion behavioural features per clip

> **Version 2**: an optimised, more robust version of the archived notebook in `notebooks/`. Changes are marked with `# [v2]` comments; see [`CHANGES.md`](../CHANGES.md).

> **How to run.** Edit only the **SETTINGS** cell below, then *Runtime → Run all*.
> On Colab, sign in to Hugging Face when asked (or add an `HF_TOKEN` secret) with an account that has
> accepted the NVIDIA dataset licence. Run the notebooks in order 1 → 5; each reads the previous one's output
> from `DATA_FOLDER`. No NVIDIA access? Try the 5-minute [demo](../demo/demo_pipeline.ipynb) instead.

In [ ]:
# ============================================================
# SETTINGS - the only cell you need to edit
# ============================================================
# DATA_FOLDER: where the pipeline_data folders (01_… to 05_…) live.
#   Colab  → a folder in your Google Drive (mounted automatically below)
#   Local  → ./pipeline_data next to the notebooks
import os, sys
IN_COLAB = "google.colab" in sys.modules
DATA_FOLDER = "/content/drive/MyDrive/driving-risk/pipeline_data" if IN_COLAB else os.path.abspath("pipeline_data")

if IN_COLAB and DATA_FOLDER.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
for sub in ("01_data_preparation", "02_ego_features", "03_visual_features", "04_dataset", "05_modelling_outputs"):
    os.makedirs(os.path.join(DATA_FOLDER, sub), exist_ok=True)
base_path = DATA_FOLDER
print("Data folder:", base_path)

In [ ]:
# ============================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ============================================================
# PURPOSE:Set up environment for data access, processing, and ML.
# OUTPUT:Required libraries installed with compatible versions.

# Install required libraries
!pip install -q huggingface_hub pyarrow scikit-learn pandas==2.2.2

# ------------------------------------------------------------
# VERIFY INSTALLATION
# ------------------------------------------------------------
import pandas as pd

print("Libraries installed successfully.\n")

print("Installed pandas version:")
print(pd.__version__)

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 1 complete: Required libraries installed and verified.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 2: AUTHENTICATE DATA ACCESS
# ============================================================
# PURPOSE:Enable access to the gated NVIDIA AV dataset.
# OUTPUT:Authenticated session for dataset download.

from huggingface_hub import login

token = os.environ.get("HF_TOKEN")              # [v2] use an HF_TOKEN secret/env var if set
if token is None and IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
login(token=token)                                # asks for a token if none is set
# ------------------------------------------------------------
# VERIFY AUTHENTICATION
# ------------------------------------------------------------
print("Login successful. Your Colab notebook is now connected to Hugging Face.")

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 2 complete: Hugging Face authentication successful.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 3: IMPORT REQUIRED LIBRARIES
# ============================================================
# PURPOSE:Load libraries for data handling, dataset access, and ML.
# OUTPUT:All required libraries imported and verified.

import pandas as pd
import numpy as np
import os

from huggingface_hub import hf_hub_download, list_repo_files

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

# ------------------------------------------------------------
# VERIFY IMPORTS
# ------------------------------------------------------------
print("Libraries imported successfully.\n")

print("Pandas version:")
print(pd.__version__)

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 3 complete: Libraries imported and environment ready.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 4: SET PROJECT CONFIGURATION
# ============================================================
# PURPOSE:Set core parameters for dataset access, sampling, and reproducibility.
# OUTPUT:Project configuration variables initialised.

repo_id = "nvidia/PhysicalAI-Autonomous-Vehicles"

sample_size_total = 2000
rare_sample_size = 1000
normal_sample_size = 1000

random_state = 42

# ------------------------------------------------------------
# VERIFY CONFIGURATION
# ------------------------------------------------------------
print("Configuration initialized successfully.\n")

print("Repository ID:")
print(repo_id)

print("\nSample size configuration:")
print("Total samples:", sample_size_total)
print("Rare samples:", rare_sample_size)
print("Normal samples:", normal_sample_size)

print("\nRandom state:")
print(random_state)

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 4 complete: Project configuration has been set.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 5: DOWNLOAD METADATA FILES
# ============================================================
# PURPOSE:Download lightweight metadata required for dataset preparation.
# OUTPUT:Metadata files downloaded and stored locally.

from huggingface_hub import hf_hub_download
import os

# ------------------------------------------------------------
# DOWNLOAD FILES
# ------------------------------------------------------------
data_collection_file = hf_hub_download(
    repo_id=repo_id,
    filename="metadata/data_collection.parquet",
    repo_type="dataset"
)

feature_presence_file = hf_hub_download(
    repo_id=repo_id,
    filename="metadata/feature_presence.parquet",
    repo_type="dataset"
)

clip_index_file = hf_hub_download(
    repo_id=repo_id,
    filename="clip_index.parquet",
    repo_type="dataset"
)

# ------------------------------------------------------------
# PRINT FILE PATHS
# ------------------------------------------------------------
print("Downloaded files successfully.\n")

print("Data Collection file saved at:")
print(data_collection_file)

print("\nFeature Presence file saved at:")
print(feature_presence_file)

print("\nClip Index file saved at:")
print(clip_index_file)

# ------------------------------------------------------------
# VERIFY FILE EXISTENCE
# ------------------------------------------------------------
print("\nFile existence check:")

print("data_collection.parquet exists:",
      os.path.exists(data_collection_file))

print("feature_presence.parquet exists:",
      os.path.exists(feature_presence_file))

print("clip_index.parquet exists:",
      os.path.exists(clip_index_file))

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 5 complete: Metadata files downloaded and verified.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 6: LOAD AND INSPECT METADATA
# ============================================================
# PURPOSE:Load metadata into DataFrames and verify structure.
# OUTPUT:DataFrames created and basic structure inspected.

import pandas as pd

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
data_collection_df = pd.read_parquet(data_collection_file)
feature_presence_df = pd.read_parquet(feature_presence_file)
clip_index_df = pd.read_parquet(clip_index_file)

# ------------------------------------------------------------
# PRINT SHAPES
# ------------------------------------------------------------
print("Data loaded successfully.\n")

print("Data Collection shape:")
print(data_collection_df.shape)

print("\nFeature Presence shape:")
print(feature_presence_df.shape)

print("\nClip Index shape:")
print(clip_index_df.shape)

# ------------------------------------------------------------
# DISPLAY SAMPLE DATA
# ------------------------------------------------------------
print("\nSample Data (Data Collection):")
display(data_collection_df.head())

print("\nSample Data (Feature Presence):")
display(feature_presence_df.head())

print("\nSample Data (Clip Index):")
display(clip_index_df.head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 6 complete: Metadata loaded and inspected.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 7: MERGE METADATA TABLES
# ============================================================
# PURPOSE:Combine collection, feature presence, and chunk information.
# OUTPUT:Single master metadata dataset created for scenario mining.

# ------------------------------------------------------------
# PREPARE CLIP INDEX
# ------------------------------------------------------------
clip_index_reset = clip_index_df.reset_index()

# ------------------------------------------------------------
# MERGE DATA COLLECTION WITH FEATURE PRESENCE
# ------------------------------------------------------------
master_df = data_collection_df.join(
    feature_presence_df,
    how="inner"
)

# ------------------------------------------------------------
# ADD CLIP INDEX INFORMATION
# ------------------------------------------------------------
master_df = master_df.reset_index()

master_df = master_df.merge(
    clip_index_reset,
    on="clip_id",
    how="inner"
)

# ------------------------------------------------------------
# PRINT MERGE RESULTS
# ------------------------------------------------------------
print("Metadata tables merged successfully.\n")

print("Master dataset shape:")
print(master_df.shape)

print("\nFirst 5 rows:")
display(master_df.head())

print("\nColumn names:")
print(list(master_df.columns))

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 7 complete: Master metadata dataset created.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 8: FILTER VALID CLIPS FOR ANALYSIS
# ============================================================
# PURPOSE:Keep only valid clips with required ego-motion and camera data.
# OUTPUT:Filtered dataset prepared for scenario mining.

# ------------------------------------------------------------
# FILTER VALID CLIPS
# ------------------------------------------------------------
filtered_df = master_df.copy()

filtered_df = filtered_df[
    (filtered_df["clip_is_valid"] == True) &
    (filtered_df["egomotion.offline"] == True) &
    (filtered_df["camera_front_wide_120fov"] == True)
].copy()

# ------------------------------------------------------------
# PRINT FILTER RESULTS
# ------------------------------------------------------------
print("Valid clips filtered successfully.\n")

print("Original master dataset shape:")
print(master_df.shape)

print("\nFiltered dataset shape:")
print(filtered_df.shape)

print("\nRows removed:")
print(master_df.shape[0] - filtered_df.shape[0])

print("\nFirst 5 rows of filtered dataset:")
display(filtered_df.head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 8 complete: Valid clips with ego-motion and camera data selected.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 9: CHECK MISSING VALUES AND BASIC DATA QUALITY
# ============================================================
# PURPOSE:Identify missing values before feature engineering.
# OUTPUT:Data quality summary for the filtered dataset.

# ------------------------------------------------------------
# CHECK MISSING VALUES
# ------------------------------------------------------------
missing_values = filtered_df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

# ------------------------------------------------------------
# PRINT DATA QUALITY SUMMARY
# ------------------------------------------------------------
print("Data quality check completed.\n")

print("Filtered dataset shape:")
print(filtered_df.shape)

print("\nColumns with missing values:")
print(missing_values)

print("\nTotal duplicated clip_id values:")
print(filtered_df["clip_id"].duplicated().sum())

print("\nBasic metadata distributions:")

print("\nTop 10 countries:")
print(filtered_df["country"].value_counts().head(10))

print("\nPlatform class distribution:")
print(filtered_df["platform_class"].value_counts())

print("\nRadar configuration distribution:")
print(filtered_df["radar_config"].value_counts(dropna=False))

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 9 complete: Missing values and basic data quality checked.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 10: PREPARE FEATURES FOR SCENARIO MINING
# ============================================================
# PURPOSE:Encode categorical variables and scale features for ML.
# OUTPUT:Numerical feature matrix ready for clustering/anomaly detection.

from sklearn.preprocessing import LabelEncoder, StandardScaler

# ------------------------------------------------------------
# SELECT FEATURES
# ------------------------------------------------------------
features_df = filtered_df[[
    "country",
    "platform_class",
    "radar_config",
    "hour_of_day",
    "month"
]].copy()

# ------------------------------------------------------------
# ENCODE CATEGORICAL VARIABLES
# ------------------------------------------------------------
le_country = LabelEncoder()
le_platform = LabelEncoder()
le_radar = LabelEncoder()

features_df["country_enc"] = le_country.fit_transform(features_df["country"].astype(str))
features_df["platform_enc"] = le_platform.fit_transform(features_df["platform_class"].astype(str))
features_df["radar_enc"] = le_radar.fit_transform(features_df["radar_config"].astype(str))

# ------------------------------------------------------------
# CREATE FINAL FEATURE MATRIX
# ------------------------------------------------------------
X = features_df[[
    "country_enc",
    "platform_enc",
    "radar_enc",
    "hour_of_day",
    "month"
]]

# ------------------------------------------------------------
# SCALE FEATURES
# ------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ------------------------------------------------------------
# PRINT FEATURE PREPARATION DETAILS
# ------------------------------------------------------------
print("Feature preparation completed.\n")

print("Feature matrix shape:")
print(X.shape)

print("\nScaled feature matrix shape:")
print(X_scaled.shape)

print("\nSample encoded features:")
display(features_df.head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 10 complete: Features encoded and scaled for scenario mining.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 11: APPLY K-MEANS CLUSTERING
# ============================================================
# PURPOSE:Group clips into scenario patterns using metadata features.
# OUTPUT:Cluster labels added to the filtered dataset.

from sklearn.cluster import KMeans

# ------------------------------------------------------------
# APPLY K-MEANS
# ------------------------------------------------------------
# Based on the earlier proof-of-concept notebook, k = 4 was
# selected as a practical cluster count using the elbow method.

kmeans = KMeans(
    n_clusters=4,
    random_state=random_state,
    n_init=10
)

filtered_df["cluster"] = kmeans.fit_predict(X_scaled)

# ------------------------------------------------------------
# PRINT CLUSTER RESULTS
# ------------------------------------------------------------
print("K-Means clustering completed.\n")

print("Cluster distribution:")
print(filtered_df["cluster"].value_counts().sort_index())

print("\nFirst 5 rows with cluster labels:")
display(filtered_df[["clip_id", "country", "hour_of_day", "month", "platform_class", "radar_config", "cluster"]].head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 11 complete: Scenario clusters assigned to valid clips.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 12: APPLY ISOLATION FOREST FOR RARE SCENARIO DETECTION
# ============================================================
# PURPOSE:Identify statistically unusual clips from metadata patterns.
# OUTPUT:Anomaly labels and anomaly scores added to the dataset.

from sklearn.ensemble import IsolationForest

# ------------------------------------------------------------
# APPLY ISOLATION FOREST
# ------------------------------------------------------------
# contamination=0.05 means the model flags approximately 5%
# of clips as rare/anomalous.

iso_forest = IsolationForest(
    contamination=0.05,
    random_state=random_state
)

filtered_df["anomaly_label"] = iso_forest.fit_predict(X_scaled)
filtered_df["anomaly_score"] = iso_forest.decision_function(X_scaled)

# ------------------------------------------------------------
# CREATE READABLE RARE/NORMAL LABEL
# ------------------------------------------------------------
filtered_df["scenario_type"] = filtered_df["anomaly_label"].map({
    -1: "rare",
    1: "normal"
})

# ------------------------------------------------------------
# PRINT ANOMALY RESULTS
# ------------------------------------------------------------
print("Isolation Forest rare scenario detection completed.\n")

print("Scenario type distribution:")
print(filtered_df["scenario_type"].value_counts())

print("\nAnomaly label distribution:")
print(filtered_df["anomaly_label"].value_counts())

print("\nAnomaly score summary:")
print(filtered_df["anomaly_score"].describe())

print("\nFirst 5 rows with anomaly outputs:")
display(filtered_df[[
    "clip_id",
    "country",
    "hour_of_day",
    "month",
    "platform_class",
    "cluster",
    "anomaly_label",
    "anomaly_score",
    "scenario_type"
]].head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 12 complete: Rare and normal scenario labels created.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 13: CREATE CHUNK-AWARE AND DIVERSITY-AWARE SAMPLE
# ============================================================
# PURPOSE:Select 2,000 clips while reducing chunk downloads and country dominance.
# OUTPUT:Balanced rare/normal sample with improved sampling efficiency and diversity.

# ------------------------------------------------------------
# SAMPLING SETTINGS
# ------------------------------------------------------------
target_total = 2000
target_rare = 1000
target_normal = 1000

max_chunks = 150
max_country_share = 0.25
# [v2] NOTE: this cap is applied separately to the rare and normal strata
#      (max 500 clips per country in each stratum), so it is a per-stratum cap,
#      not an overall cap. Report it as such.
max_per_country = int(target_total * max_country_share)

# ------------------------------------------------------------
# SEPARATE RARE AND NORMAL CLIPS
# ------------------------------------------------------------
rare_df = filtered_df[filtered_df["scenario_type"] == "rare"].copy()
normal_df = filtered_df[filtered_df["scenario_type"] == "normal"].copy()

# ------------------------------------------------------------
# FIND CHUNKS THAT CONTAIN BOTH RARE AND NORMAL CLIPS
# ------------------------------------------------------------
chunk_summary = filtered_df.groupby("chunk").agg(
    total_clips=("clip_id", "count"),
    rare_clips=("scenario_type", lambda x: (x == "rare").sum()),
    normal_clips=("scenario_type", lambda x: (x == "normal").sum()),
    country_count=("country", "nunique")
).reset_index()

chunk_summary["balanced_score"] = (
    chunk_summary["rare_clips"] +
    chunk_summary["normal_clips"] +
    chunk_summary["country_count"]
)

candidate_chunks = chunk_summary[
    (chunk_summary["rare_clips"] > 0) &
    (chunk_summary["normal_clips"] > 0)
].sort_values(
    by="balanced_score",
    ascending=False
).head(max_chunks)["chunk"].tolist()

# ------------------------------------------------------------
# FILTER DATA TO SELECTED CHUNKS
# ------------------------------------------------------------
candidate_df = filtered_df[
    filtered_df["chunk"].isin(candidate_chunks)
].copy()

# ------------------------------------------------------------
# COUNTRY-CAPPED SAMPLING FUNCTION
# ------------------------------------------------------------
def country_capped_sample(df, n_target, max_per_country, random_state=42):
    sampled_parts = []

    for country, group in df.groupby("country"):
        n_country = min(len(group), max_per_country)

        sampled_parts.append(
            group.sample(
                n=n_country,
                random_state=random_state
            )
        )

    sampled_df = pd.concat(sampled_parts, axis=0)

    if len(sampled_df) > n_target:
        sampled_df = sampled_df.sample(
            n=n_target,
            random_state=random_state
        )

    return sampled_df

# ------------------------------------------------------------
# SAMPLE RARE AND NORMAL CLIPS
# ------------------------------------------------------------
rare_candidate_df = candidate_df[candidate_df["scenario_type"] == "rare"].copy()
normal_candidate_df = candidate_df[candidate_df["scenario_type"] == "normal"].copy()

rare_sample_df = country_capped_sample(
    rare_candidate_df,
    target_rare,
    max_per_country,
    random_state
)

normal_sample_df = country_capped_sample(
    normal_candidate_df,
    target_normal,
    max_per_country,
    random_state
)

# ------------------------------------------------------------
# COMBINE AND SHUFFLE FINAL SAMPLE
# ------------------------------------------------------------
prepared_dataset = pd.concat(
    [rare_sample_df, normal_sample_df],
    axis=0
).sample(
    frac=1,
    random_state=random_state
).reset_index(drop=True)

# ------------------------------------------------------------
# PRINT SAMPLE RESULTS
# ------------------------------------------------------------
print("Chunk-aware and diversity-aware sample created successfully.\n")

print("Selected chunk count:")
print(len(candidate_chunks))

print("\nPrepared dataset shape:")
print(prepared_dataset.shape)

print("\nScenario type distribution:")
print(prepared_dataset["scenario_type"].value_counts())

print("\nUnique chunks in final sample:")
print(prepared_dataset["chunk"].nunique())

print("\nCountry distribution top 10:")
print(prepared_dataset["country"].value_counts().head(10))

print("\nCountry share top 10:")
print((prepared_dataset["country"].value_counts(normalize=True).head(10) * 100).round(2))

print("\nDay/night distribution preview:")
print(prepared_dataset["hour_of_day"].apply(lambda x: "night" if (x >= 20 or x <= 6) else "day").value_counts())

print("\nFirst 5 rows:")
display(prepared_dataset.head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 13 complete: Chunk-aware and diversity-aware 2,000-clip sample created.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 14: ADD DIVERSITY FEATURES FOR DATASET REVIEW
# ============================================================
# PURPOSE:Create day/night and peak-hour indicators for reporting.
# OUTPUT:Prepared dataset enriched with context features.

# ------------------------------------------------------------
# CREATE DAY / NIGHT FEATURE
# ------------------------------------------------------------
prepared_dataset["is_night"] = prepared_dataset["hour_of_day"].apply(
    lambda x: 1 if (x >= 20 or x <= 6) else 0
)

# ------------------------------------------------------------
# CREATE PEAK-HOUR FEATURE
# ------------------------------------------------------------
prepared_dataset["is_peak_hour"] = prepared_dataset["hour_of_day"].apply(
    lambda x: 1 if ((7 <= x <= 10) or (16 <= x <= 19)) else 0
)

# ------------------------------------------------------------
# PRINT DIVERSITY SUMMARY
# ------------------------------------------------------------
print("Diversity features created successfully.\n")

print("Prepared dataset shape:")
print(prepared_dataset.shape)

print("\nScenario type distribution:")
print(prepared_dataset["scenario_type"].value_counts())

print("\nUnique chunks:")
print(prepared_dataset["chunk"].nunique())

print("\nDay/night distribution:")
print(prepared_dataset["is_night"].value_counts())

print("\nPeak-hour distribution:")
print(prepared_dataset["is_peak_hour"].value_counts())

print("\nCountry distribution top 10:")
print(prepared_dataset["country"].value_counts().head(10))

print("\nFirst 5 rows with diversity features:")
display(prepared_dataset[[
    "clip_id",
    "country",
    "hour_of_day",
    "is_night",
    "is_peak_hour",
    "scenario_type",
    "chunk"
]].head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 14 complete: Diversity features added and reviewed.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 15: SELECT FINAL COLUMNS FOR NOTEBOOK 1 OUTPUT
# ============================================================
# PURPOSE:Keep only columns needed for later feature extraction and modelling.
#         anomaly_score is intentionally EXCLUDED from output - it is used
#         for sampling only and must NOT be used as a label or feature.
#         Risk labels will be created in Notebook 5 on training data only.
# OUTPUT:Clean prepared dataset ready for export.

# ------------------------------------------------------------
# DEFINE COLUMNS TO KEEP
# ------------------------------------------------------------
columns_to_keep = [
    "clip_id",
    "chunk",
    "split",
    "clip_is_valid",

    "country",
    "month",
    "hour_of_day",
    "platform_class",
    "radar_config",

    "camera_front_wide_120fov",
    "egomotion.offline",

    "cluster",
    "anomaly_label",
    "scenario_type",

    "is_night",
    "is_peak_hour"

    # NOTE: anomaly_score deliberately excluded.
    # It was used for rare/normal sampling only (Isolation Forest contamination=0.05).
    # Final risk labels are created in Notebook 5 on training data only
    # to prevent target leakage.
]

# ------------------------------------------------------------
# CREATE CLEAN OUTPUT DATASET
# ------------------------------------------------------------
prepared_dataset_clean = prepared_dataset[columns_to_keep].copy()

# ------------------------------------------------------------
# PRINT OUTPUT DETAILS
# ------------------------------------------------------------
print("Final columns selected successfully.\n")

print("Prepared clean dataset shape:")
print(prepared_dataset_clean.shape)

print("\nUnique chunks:")
print(prepared_dataset_clean["chunk"].nunique())

print("\nScenario type distribution:")
print(prepared_dataset_clean["scenario_type"].value_counts())

print("\nColumns included:")
print(list(prepared_dataset_clean.columns))

print("\nFirst 5 rows:")
display(prepared_dataset_clean.head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 15 complete: Clean prepared dataset created for export.")
print("NOTE: anomaly_score excluded - risk labels created in Notebook 5.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 16: SAVE PREPARED DATASET TO GOOGLE DRIVE
# ============================================================
# PURPOSE:Persist Notebook 1 output for reuse in later notebooks.
# OUTPUT:CSV and Parquet files saved to Google Drive.

import os

# ------------------------------------------------------------
# MOUNT GOOGLE DRIVE
# ------------------------------------------------------------
# ------------------------------------------------------------
# CREATE PROJECT FOLDER
# ------------------------------------------------------------
# base_path comes from the SETTINGS cell at the top
notebook1_path = f"{base_path}/01_data_preparation"

os.makedirs(notebook1_path, exist_ok=True)

# ------------------------------------------------------------
# DEFINE OUTPUT FILE PATHS
# ------------------------------------------------------------
csv_output_path = f"{notebook1_path}/prepared_dataset_2000.csv"
parquet_output_path = f"{notebook1_path}/prepared_dataset_2000.parquet"

# ------------------------------------------------------------
# SAVE FILES
# ------------------------------------------------------------
prepared_dataset_clean.to_csv(csv_output_path, index=False)
prepared_dataset_clean.to_parquet(parquet_output_path, index=False)

# ------------------------------------------------------------
# VERIFY SAVED FILES
# ------------------------------------------------------------
print("Prepared dataset saved successfully.\n")

print("CSV file saved at:")
print(csv_output_path)

print("\nParquet file saved at:")
print(parquet_output_path)

print("\nFile existence check:")
print("CSV exists:", os.path.exists(csv_output_path))
print("Parquet exists:", os.path.exists(parquet_output_path))

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 16 complete: Prepared dataset saved to Google Drive.")
print("------------------------------------------------------------")